# 11 · Controller, baselines, headline (stages S11-S14)

The grid already holds every `(k, R)` answer of every item, so no model runs here. A controller path (exit, loop again,
or look again = restart with all visual tokens) is one grid cell plus the work thrown away, so we can build every path of
every item (the tree), train the controller on those trees, run the baselines on the same items and compare accuracy at
the same mean FLOPs.

**Who picks what:** weights (and the heuristic's thresholds) come from `ctrl_train`; the epoch and the architecture from
`ctrl_val`, 20% of the `ctrl_train` images held out; only `(k0, lambda)` per cost cap from dev. Dev numbers are
**preliminary** until test. The test split stays sealed until `configs/prereg.yaml` exists.

**How to run:** Run All after the full `grid.parquet` is in the grid folder (`GRID_PADDED` names it). CPU only.
The controller step trains every capacity variant (about 15 minutes at most on the full grid).
The two upfront baselines also need `python scripts/mvp.py upfront` (GPUs 0-3, one model copy); without those files
they are skipped with a printed line.
`SYNTHETIC = True` runs everything on a made-up grid under `execution_results/synthetic_nb11/` (a code check; its
numbers mean nothing). A `CUDA_VISIBLE_DEVICES` warning from `setup_notebook` is expected: no GPU is used.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import socket

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only; before adaloop.api is imported

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
ctrl = api.load_config(ROOT / "configs/controller.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())
print("grid config_hash", api.config_hash(api.grid_config(cfg)))  # what the grid rows carry

In [ ]:
print("host", socket.gethostname(), "| on Inara or PACE:", api.is_gpu_host())

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Settings (values: configs/controller.yaml)
SYNTHETIC = False  # True: made-up grid, nothing is written to the real output folders
FINAL = False  # figure to paper/: only for test results
GRID_PADDED = True  # the PACE grid in <out_dir>/grid_b8_padded; False: Inara's equal-length grid_b8
GRID_FILE = "grid.parquet"  # "grid_partial.parquet": smoke test on a quick pass; outputs go to controller_partial/
if GRID_PADDED:
    cfg = {**cfg, "batch_grouping": None}  # placement only (not hashed): it names the grid folder
if SYNTHETIC:
    cfg = api.write_synthetic_grid(cfg, ctrl, ROOT)  # same file layout, other out_dir
    ctrl = {**ctrl, "epochs": ctrl["synthetic"]["epochs"],
            "bootstrap": {**ctrl["bootstrap"], "n_boot": ctrl["synthetic"]["n_boot"]}}
    params = api.SYNTHETIC_PARAMS
else:
    params = api.cost_params_from_config(cfg["model_id"])  # meta device: config only, no weights
if GRID_FILE != "grid.parquet" and not SYNTHETIC:
    ctrl = {**ctrl, "grid_file": GRID_FILE, "out_dir": "controller_partial"}

K0, LOOK_K = ctrl["k0"], ctrl["look"]["k"]
LAMBDAS, SEEDS = ctrl["lambdas"], ctrl["seeds"]
HEADLINE = ctrl["headline_frac"]
LABEL = api.split_label(ctrl, api.report_split(ctrl))
OUT = api.controller_dir(cfg, ctrl, ROOT)
assert not (FINAL and GRID_FILE != "grid.parquet"), "FINAL needs the full grid"
assert not (FINAL and (SYNTHETIC or api.report_split(ctrl) != "test")), "FINAL figures come from test results only"
print("k0", K0, "| lambdas", LAMBDAS, "| seeds", SEEDS, "| headline", HEADLINE, "x cost_ref |", LABEL)
print("grid", api.grid_dir(cfg, ROOT, ctrl["grid_name"]))
print("out", OUT)

### Step-1:- The trajectory tree (S11)

For every item and first keep fraction `k0`: exit after pass 1..5, or look after pass 1..4 and then exit after pass 1..5
of the full-token restart (25 leaves). A leaf copies its answer from the grid and adds up its FLOPs.

In [ ]:
grid, features = api.load_controller_inputs(cfg, ctrl, ROOT)  # main run, one model variant
grid.drop_duplicates(["benchmark", "item_id"]).groupby(["fit_split", "benchmark"]).size()

`fit_split` is `split` with the items of the held-out `ctrl_train` images renamed `ctrl_val` (no new inference).

In [ ]:
images = api.load_item_images(cfg, ctrl, ROOT)
seen = grid.assign(image=[images[b][i] for b, i in zip(grid.benchmark, grid.item_id)])
side = {name: set(zip(part.benchmark, part.image)) for name, part in seen.groupby("fit_split")}
assert not side["ctrl_val"] & side["ctrl_train"]
print({name: len(v) for name, v in side.items()}, "images")
print("✓ PASS S12 splits: ctrl_val shares no image with ctrl_train")

#### One item first

In [ ]:
first = grid.iloc[0]
one = grid[(grid.benchmark == first.benchmark) & (grid.item_id == first.item_id)]
sample_tree = api.build_tree(one, features, params, ctrl)
sample_tree[sample_tree.k0 == K0[0]][["path", "correct", "flops_total", "flops_cell", "flops_discarded", "flops_readout", "flops_state"]]

In [ ]:
leaf = sample_tree[(sample_tree.k0 == K0[0]) & (sample_tree.path == "X@3")].iloc[0]
cell = one[(one.k == K0[0]) & (one.R == 3)].iloc[0]
assert leaf.correct == cell.correct and leaf.flops_cell == cell.flops_total
print("✓ PASS S11 tree: the leaf X@3 is the grid cell (k0, R=3)")

#### The pass features of that item

Besides the saved scalars, two are read from the saved top-32 log-probs: `top2_margin` (p1 - p2 of the first answer token)
and `pred_changed` (the most likely token differs from the previous pass; NaN at pass 1). Both need the early readout, so
they are charged like entropy.

In [ ]:
passes = api.top_features(features[first.benchmark, first.item_id])
names = ["delta_norm", "kl_prev", "entropy", "top1_prob", "vis_attn_mass", "top2_margin", "pred_changed"]
shown = pd.DataFrame({p: passes[K0[0], p] for p in range(1, int(grid.R.max()) + 1)}).T[names].astype(float)
shown

In [ ]:
assert np.isnan(shown.pred_changed.iloc[0]) and shown.pred_changed.iloc[1:].isin([0.0, 1.0]).all()
assert shown.top2_margin.between(0, 1).all() and (shown.top2_margin <= shown.top1_prob + 1e-4).all()
print("✓ PASS S12 features: margin and changed-token flag from the saved top-32")

#### All items

In [ ]:
print("[STAGE] trees of all items")
tree, features = api.run_tree(cfg, ctrl, ROOT, params)

In [ ]:
R_max = int(grid.R.max())
assert len(tree) == len(features) * len(K0) * (R_max + (R_max - 1) * R_max)
assert (tree.flops_total >= tree.flops_cell).all() and "test" not in set(tree.split)
print(f"✓ PASS S11 tree: {len(tree)} leaves, costs include the cell, no test rows")

### Step-2:- The controller and the heuristic (S12)

A small network reads the pass state (`h_last` and scalar features) and picks EXIT, LOOP or LOOK. It trains on `ctrl_train`
trees with reward `correct - lambda * cost / cost_ref`; the epoch is chosen on `ctrl_val`. The heuristic searches
thresholds on three of the features instead, on all of `ctrl_train`.

#### One training run first

In [ ]:
train = api.tree_data(tree, features, K0[0], LOOK_K, ctrl["splits"]["train"])
val = api.tree_data(tree, features, K0[0], LOOK_K, ctrl["splits"]["val"])
dev = api.tree_data(tree, features, K0[0], LOOK_K, ctrl["splits"]["dev"])
policy, norm, info = api.train_policy(train, val, LAMBDAS[1], SEEDS[0], {**ctrl, "epochs": 2})
exits = api.greedy_exits(policy, dev, norm, ctrl)
api.policy_rows(dev, exits, "controller", LAMBDAS[1], SEEDS[0], True, True).head()

#### Every k0, lambda and seed

In [ ]:
print("[STAGE] controller and heuristic")
rows = api.run_policies(cfg, ctrl, ROOT, params)  # writes tree.parquet, policy_eval.parquet, weights
rows.groupby(["method", "k0", "lam"])[["correct", "flops_total"]].mean().head(12)

In [ ]:
n_dev = len(dev.keys)
sizes = rows[rows.split == "dev"].groupby(["method", "k0", "lam", "seed"]).size()
assert (sizes == n_dev).all() and len(sizes) == len(K0) * len(LAMBDAS) * (len(SEEDS) + 1)
print(f"✓ PASS S12 controller: {len(sizes)} settings, each with one row per dev item ({n_dev})")

#### Capacity: which architecture?

A few hundred training items against `h_last` projected to 256 can overfit. `run_policies` trained every variant of
`capacity` (with / without `h_last`, projection 256 or 32, MLP or linear head); the best mean `ctrl_val` reward chose the
`controller`. The best architecture is also trained with `node_weight: reach` (each node of the loss weighted by the
probability of reaching it; the default `uniform` is the spec's), again judged on `ctrl_val`. All variants are listed;
their dev numbers choose nothing.

In [ ]:
capacity = pd.read_csv(OUT / "tables" / "T4_capacity.csv")
ARCH = api.chosen_arch(OUT)
capacity.round(4)

In [ ]:
best = capacity.set_index("variant").val_reward
assert len(capacity) == len(ctrl["capacity"]) + len(ctrl["node_weight_options"]) - 1 and capacity.chosen.sum() == 1
assert best[capacity.set_index("variant").chosen].iloc[0] == best.max()
print(f"✓ PASS S12 capacity: {capacity[capacity.chosen].variant.iloc[0]} chosen on ctrl_val | {ARCH}")

#### Has the controller collapsed?

A controller that sends every item down one path has learned nothing per item. Per `(k0, lambda)` on dev: the share of
EXIT / LOOP / LOOK over the visited nodes, the exit pass, the share of items that look and the mean cost, for the
controller and for the oracle (the best action at every node, read from the labels). `collapsed`: one path takes more
than `collapse_share` of the items in some seed while the oracle does not.

In [ ]:
collapse = api.collapse_table(rows, tree, features, ctrl)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
collapse.to_csv(OUT / "tables" / "T5_collapse.csv", index=False)
print(collapse.round(2).to_string(index=False))

In [ ]:
assert np.allclose(collapse[["EXIT", "LOOP", "LOOK"]].sum(axis=1), 100)
flagged = collapse[collapse.collapsed]
print(f"{len(flagged)} of {len(collapse) // 2} (k0, lambda) settings collapsed")
print(f"✓ PASS S12 diagnostics: action shares, exit passes and cost for controller and oracle")

### Step-3:- Baselines (S13)

Same items, same costs, all written into `policy_eval.parquet`.

#### Best fixed setting and oracle, per cost cap

`best_fixed`: no per-item information. Each item gets one of two neighbouring cells of the upper hull at random, so the
mean cost is exactly the cap; we use the expected score (nothing is drawn). Hull from `ctrl_train` accuracy and the costs
of the evaluated items. Above its most accurate cell it stays there and spends less than the cap.
`best_fixed_cell` (secondary, the prereg wording): one `(k, R)` for every item with mean FLOPs <= cap.
`oracle`: each item its own cell with the mean cost under the cap; it reads the labels, so it is only an upper bound.

In [ ]:
best = api.best_fixed_rows(grid, ctrl)
single = api.best_fixed_cell_rows(grid, ctrl)
oracle = api.oracle_rows(grid, ctrl)
pd.concat([best, single, oracle]).groupby(["method", "cap_frac"])[["correct", "flops_total"]].mean()

In [ ]:
for part in (best, single, oracle):
    spent = part.groupby("cap_frac")[["flops_total", "cost_ref"]].mean()
    assert (spent.flops_total <= spent.index * spent.cost_ref * (1 + 1e-9) + 1).all()  # + 1: whole FLOPs per row
both = single.groupby("cap_frac").correct.mean().to_frame("single").join(oracle.groupby("cap_frac").correct.mean())
assert (both.correct >= both.single - 1e-12).all()
print("✓ PASS S13 caps: mean FLOPs within every cap, oracle >= best fixed cell")

#### Early exit only, and the controller without h_last

`early_exit_only`: all tokens, EXIT or LOOP, same architecture (`ARCH`, chosen on `ctrl_val`) and features.
`controller_no_h`: that architecture with scalar features only (the same as the controller if a scalar variant was chosen).

In [ ]:
print("[STAGE] early_exit_only")
early = api.early_exit_rows(grid, features, params, {**ctrl, **ARCH}, weights_dir=OUT / "weights")
print("[STAGE] controller_no_h")
no_h = api.no_h_rows(tree, features, {**ctrl, **ARCH}, weights_dir=OUT / "weights")

In [ ]:
assert not early.path.str.startswith("L@").any() and set(early.k0) == {LOOK_K}
assert len(no_h) == (rows.method == "controller").sum()
print("✓ PASS S13 early exit never looks; ablation covers the same settings as the controller")

#### Upfront baselines: choose before decoding

Input: the mean visual token and the mean question embedding (vision tower and embedding table, no decoder pass).
`tokens_only` picks k at R = 3; `upfront_15way` picks one of the 15 cells. Each is charged its cell (which includes the
vision tower) plus the two pooling sums.

In [ ]:
upfront = api.load_upfront(grid, api.upfront_dir(cfg, ROOT, ctrl["grid_name"]))  # None: files not made yet
upfront_parts = []
if upfront is not None:
    print("[STAGE] tokens_only and upfront_15way")
    upfront_parts = [api.tokens_only_rows(grid, upfront, ctrl), api.upfront_15way_rows(grid, upfront, ctrl)]
else:
    print("SKIPPED tokens_only and upfront_15way: no upfront features")

In [ ]:
if upfront is not None:
    assert upfront_parts[0].path.str.endswith("@3").all()
    assert all(len(p) == len(LAMBDAS) * len(SEEDS) * n_dev for p in upfront_parts)
    print("✓ PASS S13 upfront: tokens_only stays at R=3; one row per lambda, seed and dev item")

#### Random controller

The controller's own paths dealt to random items, so only the matching of path to item is lost. No feature costs.

In [ ]:
random = api.random_rows(tree, rows[rows.method == "controller"], ctrl["baselines"]["random_seed"])
rows = api.save_baseline_rows(OUT / "policy_eval.parquet", [best, single, oracle, early, no_h, random, *upfront_parts])

In [ ]:
key = ["k0", "lam", "seed", "path"]
paths = [rows[rows.method == m].groupby(key).size().sort_index() for m in ("controller", "random")]
assert paths[0].equals(paths[1]) and "test" not in set(rows.split)
print("✓ PASS S13 random: same paths as the controller, other items")

### Step-4:- Headline (S14)

Each method gives a curve (accuracy against mean FLOPs over lambda). The headline reads every curve at
`HEADLINE x cost_ref` by a straight line between the two neighbouring points; a curve that does not get there gives NaN
(no extrapolation). `acc_within_cap` is the best dev-chosen point under the cap. `best_fixed` (hull mix, at exactly the
cap), `best_fixed_cell` and `oracle` are the values of that cap itself.

**Pending Vedaang:** the headline comparison (controller against `best_fixed` read as the hull mix, not the single cell of
the prereg wording) is not confirmed yet. Do not quote it before that.

In [ ]:
print("[STAGE] curves, bootstrap")
results = api.run_headline(rows, ctrl, OUT)
print("PENDING VEDAANG: headline comparison (best_fixed = hull mix of fixed cells) is not confirmed")
results["T1_headline"].round(2)

#### Controller against each baseline

Paired bootstrap over items; Holm over the claims in the config. Other rows are exploratory.

In [ ]:
results["T3_comparisons"].round(4)

In [ ]:
main = results["T3_comparisons"].set_index("baseline").loc["best_fixed"]
met = bool(main["diff"] >= ctrl["min_gain_pts"] and main.ci_low > 0)
print(f"PENDING VEDAANG | {LABEL}: controller - best_fixed (hull mix) {main['diff']:.2f} pts [{main.ci_low:.2f}, {main.ci_high:.2f}] -> "
      f"preregistered rule {'met' if met else 'not met'} (a preview, not the verdict)")

#### Every cost cap

In [ ]:
results["T3_by_cap"].pivot(index="cap_frac", columns="method", values="accuracy").round(1)

#### Headroom gate

`oracle - best_fixed - noise` per benchmark. `best_fixed` is the hull mix, built per benchmark like the headline's;
`headroom` and `met` use it. The `_cell` columns use the single best cell, as nb08 does, for comparison. Noise is the larger of two measurements: a re-run of the same setting, and
batch 1 against batch 8 (nb08 Step-2b). Step-2b ran only k = 25% and 100% at R = 3 on dev, so the batch noise sees 2 of
the 15 cells and can understate it.

In [ ]:
gate = api.headroom_gate(cfg, ROOT, ctrl["variant"], ctrl["grid_name"], ctrl.get("grid_file", "grid.parquet"))
gate.to_csv(OUT / "tables" / "T2_headroom_gate.csv", index=False)
print("Limitation: batch noise from nb08 Step-2b covers k = 25% and 100% at R = 3 only")
for row in gate.itertuples():
    print(f"{row.benchmark}: headroom {row.headroom:.2f} pts over the hull mix | {row.headroom_cell:.2f} over the single cell")
display(gate.round(2).T)

#### Headroom per lambda

The gate above is one cost. Here, at the cost of every controller point: `headroom = oracle - best_fixed (hull) - noise`
next to `gain = controller - best_fixed`. Little headroom: a flat or collapsed controller has nothing to find there.
Headroom with no gain: the controller is the weak part. `oracle` reads the labels.

In [ ]:
by_lambda = api.headroom_by_lambda(rows, collapse, cfg, ctrl, ROOT)
by_lambda.to_csv(OUT / "tables" / "T6_headroom_by_lambda.csv", index=False)
for p in by_lambda.itertuples():
    note = "little headroom" if p.headroom < cfg["headroom"]["min_pts"] else "headroom"
    note += " | collapsed" if p.collapsed else ""
    print(f"k0={p.k0} lam={p.lam:<4} cost {p.cost_frac:.2f} | controller {p.controller:.1f} | best_fixed {p.best_fixed:.1f} "
          f"| gain {p.gain:+.1f} | oracle {p.oracle:.1f} | noise {p.noise:.1f} | headroom {p.headroom:+.1f} | {note}")

In [ ]:
assert len(by_lambda) == len(K0) * len(LAMBDAS) and (by_lambda.oracle.dropna() >= 0).all()
print(f"✓ PASS S14 headroom per lambda: {len(by_lambda)} controller points, {LABEL}")

### Step-5:- Figure F10

In [ ]:
api.set_paper_style()
title = ("SYNTHETIC | " if SYNTHETIC else "") + LABEL
fig = api.plot_iso_cost(results["F10_iso_cost"], HEADLINE, title)
if SYNTHETIC:
    print("synthetic run: figure not saved")
else:
    api.save_figure(fig, "F10_iso_cost", final=FINAL)
fig

In [ ]:
assert (OUT / "tables" / "T1_headline.csv").is_file() and (OUT / "figures_data" / "F10_iso_cost.parquet").is_file()
table = results["T1_headline"].set_index("method")
assert (table.accuracy.notna() | table.note.str.len().gt(0)).all()
print(f"✓ PASS S14 headline: tables in {OUT / 'tables'}; every method has a number or a stated reason")